# CCPS-URL — Four-corpus definitive study
**Scientia Iranica, Transactions D** · Source-only feature selection and calibration · Four-corpus leave-one-corpus-out.

**All outputs will be generated from your attached real datasets.** The research method has not yet demonstrated a performance advantage.

Required Kaggle inputs: PhiUSIIL CSV or `PhishRoute_results.zip`; URL-Phish V2 CSV; LegitPhish V2 CSV; **CompPhish V4 `Mapping_File.xlsx`**. Do **not** download the large HTML ZIP.

Turn Kaggle Internet ON if tldextract needs installing; all detected input paths and SHA-256 hashes are recorded.

In [ ]:
# Install ONLY if missing. The Public Suffix List is the frozen snapshot bundled with tldextract.
import sys, subprocess, importlib
try:
    import tldextract
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'tldextract==5.3.0'])
import pandas, sklearn, scipy, openpyxl
print('Runtime versions:', 'pandas',pandas.__version__,'sklearn',sklearn.__version__,'scipy',scipy.__version__)

## File paths
The default `None` setting searches `/kaggle/input` automatically for the expected filenames. If a dataset is not detected, set its exact path below. A `Mapping_File.xlsx` can be added directly without any HTML archive.

In [ ]:
# USER CONFIGURATION (edit only if automatic discovery fails)
DATA_PATHS = {
    'PhiUSIIL': None,      # e.g. '/kaggle/input/phiusiil/PhiUSIIL_Phishing_URL_Dataset.csv'
    'URL-Phish': None,     # e.g. '/kaggle/input/url-phish/URL-Phish.csv'
    'LegitPhish': None,    # e.g. '/kaggle/input/legitphish/LegitPhish Dataset.csv'
    'CompPhish': None,     # e.g. '/kaggle/input/compphish/Mapping_File.xlsx'
}
OUTPUT_DIR='/kaggle/working/CCPS_results'
SEED_FIXED=42

## Complete source code
This cell is self-contained: no external `.py` attachment is needed. It does not use target-corpus labels for training or calibration.

In [ ]:
#!/usr/bin/env python3
"""CCPS-URL: source-only, leave-one-corpus-out reproducible experiment.

Research prototype. Does not guarantee external FPR or establish novelty.
Labels are standardized to 1=phishing; no target labels/features used in fitting,
feature selection, tuning or threshold calibration.
"""
from __future__ import annotations
import os, re, io, json, csv, math, hashlib, zipfile, itertools, time, argparse
from pathlib import Path
from urllib.parse import urlsplit
from collections import Counter
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp, beta
from sklearn.metrics import roc_auc_score, matthews_corrcoef, confusion_matrix, average_precision_score
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import HistGradientBoostingClassifier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

CORPORA=['PhiUSIIL','URL-Phish','LegitPhish','CompPhish']
SEED=42
CAL_FPR=0.01
MAX_CLASS_SOURCE=12500
MAX_CLASS_DOMAIN=50
KS_SAMPLE=4000
MI_SAMPLE=25000
MIN_TRAIN_CLASS=80
MIN_EVAL_CLASS=100
MIN_EVAL_GROUPS=15
MIN_RETENTION=0.10
BOOT_REPS=400
K_VALUES=(8,16,24)
LAMBDA_VALUES=(0.0,0.1,0.3)
F_NAMES=['url_len','host_len','path_len','query_len','fragment_len','domain_dots',
         'subdomain_count','digit_count','digit_ratio','letter_ratio','hyphen_count',
         'dot_count','at_count','equal_count','question_count','amp_count',
         'percent_count','slash_count','underscore_count','tilde_count','colon_count',
         'hash_count','semicolon_count','is_https','is_ip','punycode','has_port',
         'url_entropy','host_entropy','token_count','host_hyphen_count']

DATA_PATTERNS={
 'PhiUSIIL':['PhiUSIIL_Phishing_URL_Dataset.csv','PhiUSIIL_source.zip','PhishRoute_results.zip'],
 'URL-Phish':['URL-Phish_Dataset.csv','URL-Phish.csv','URL-Phish_Dataset.zip','Dataset.csv'],
 'LegitPhish':['LegitPhish Dataset.csv','url_features_extracted1.csv','LegitPhish.csv'],
 'CompPhish':['Mapping_File.xlsx','Mapping_File.csv'],
}
SOURCE_BAD_LABEL={'PhiUSIIL':0,'URL-Phish':1,'LegitPhish':0,'CompPhish':1}

def sha256(path):
 h=hashlib.sha256()
 with open(path,'rb') as fh:
  for buf in iter(lambda:fh.read(1024*1024),b''):h.update(buf)
 return h.hexdigest()

def candidates(roots):
 for root in roots:
  p=Path(root)
  if not p.exists():continue
  if p.is_file():yield p;continue
  for ext in ('*.csv','*.xlsx','*.zip'):
   yield from p.rglob(ext)

def locate(name, supplied, roots):
 if supplied:
  p=Path(supplied)
  if not p.is_file():raise FileNotFoundError(f'{name}: explicit input missing: {p}')
  return p
 allowed={s.lower() for s in DATA_PATTERNS[name]}
 hits=[]
 for p in candidates(roots):
  low=str(p).lower(); n=p.name.lower()
  if n not in allowed:continue
  if name=='URL-Phish' and n=='dataset.csv' and not any(s in low for s in ['url-phish','urlphish','65z9twcx3r']):continue
  hits.append(p)
 if not hits:raise FileNotFoundError(
  f'{name}: dataset not found. Add original dataset to Kaggle > Add Input; '
  f'accepted filenames: {DATA_PATTERNS[name]}. Explicit --{name.lower().replace("-","")} is supported.')
 hits.sort(key=lambda x:(len(str(x)),str(x)))
 if len(hits)>1:
  print(f'NOTE multiple possible {name} files:',[str(x) for x in hits[:5]],'\nChoosing',hits[0])
 return hits[0]

def read_raw(path,name):
 p=Path(path)
 if p.suffix.lower()=='.xlsx':
  for header in range(6):
   df=pd.read_excel(p,header=header,engine='openpyxl')
   c={re.sub(r'[^a-z0-9]+','',str(x).lower()) for x in df.columns}
   if any(x in c for x in ('url','websiteurl','weburl','link','webaddress')) and any(x in c for x in ('label','classlabel','class','target','status')):
    return df,str(p),sha256(p)
  raise ValueError(f'{name}: Mapping_File.xlsx does not expose recognizable URL+label columns. Inspect its header.')
 if p.suffix.lower()=='.csv':return pd.read_csv(p,low_memory=False),str(p),sha256(p)
 if p.suffix.lower()=='.zip':
  with zipfile.ZipFile(p) as z:
   want=[n for n in z.namelist() if n.lower().endswith('.csv') and 'phiusiil_phishing_url_dataset' in n.lower()]
   if name!='PhiUSIIL' or not want:raise ValueError(f'Unsupported ZIP content for {name}: {p}; extract original CSV/XLSX and attach it.')
   raw=z.read(want[0]);return pd.read_csv(io.BytesIO(raw),low_memory=False),f'{p}!{want[0]}',hashlib.sha256(raw).hexdigest()
 raise ValueError(f'Unexpected input type {p.suffix}')

def unify(df,name):
 columns={re.sub(r'[^a-z0-9]+','',str(c).strip().lower()):c for c in df.columns}
 u=next((columns[k] for k in ['url','websiteurl','weburl','link','webaddress','website','urladdress'] if k in columns),None)
 l=next((columns[k] for k in ['label','classlabel','target','class','status'] if k in columns),None)
 if not u or not l:raise ValueError(f'{name}: URL or label column not found. Available columns: {df.columns.tolist()}')
 part=df[[u,l]].copy();part.columns=['URL','source_label']
 part['URL']=part['URL'].astype('string').str.strip()
 labels=part.source_label.astype(str).str.strip().str.lower()
 # Explicitly map semantic string labels. Numbers follow the source's published convention.
 mapped=labels.map({'phishing':1,'phish':1,'malicious':1,'legitimate':0,'legit':0,'benign':0,'safe':0,'0':int(SOURCE_BAD_LABEL[name]==0),'1':int(SOURCE_BAD_LABEL[name]==1),'0.0':int(SOURCE_BAD_LABEL[name]==0),'1.0':int(SOURCE_BAD_LABEL[name]==1)})
 original=len(part)
 valid=part.URL.notna() & part.URL.str.len().gt(3) & mapped.notna()
 part=part[valid].copy();part['y']=mapped[valid].astype('int8');part=part.drop(columns=['source_label'])
 # URL string is left intact for lexical features; only trim whitespace.
 conflict=part.groupby('URL',sort=False).y.nunique()
 bad=set(conflict[conflict>1].index)
 n_bad=int(part.URL.isin(bad).sum());part=part[~part.URL.isin(bad)]
 part=part.drop_duplicates('URL').reset_index(drop=True)
 info={'corpus':name,'input_rows':int(original),'valid_rows':int(valid.sum()),'conflicting_label_rows':n_bad,'deduplicated_rows':int(len(part)), 'n_legitimate':int((part.y==0).sum()),'n_phishing':int((part.y==1).sum()),'url_column':str(u),'label_column':str(l)}
 if min(info['n_legitimate'],info['n_phishing'])<MIN_TRAIN_CLASS:raise ValueError(f'{name}: missing class or ambiguous labels: {info}')
 return part,info

def parse_url(raw):
 raw=str(raw).strip()
 if not re.match(r'^[a-zA-Z][a-zA-Z0-9+.-]*://',raw): raw='http://'+raw
 try:
  p=urlsplit(raw); host=(p.hostname or '').lower().strip('.')
 except (ValueError,OverflowError):
  p=urlsplit('http://invalid.local/');host='invalid.local'
 return p,host

def shannon(s):
 if not s:return 0.
 c=Counter(s);n=len(s)
 return -sum((v/n)*math.log2(v/n) for v in c.values())

def feature_row(url,prefix_ablation=False):
 original=str(url).strip();p,host=parse_url(original)
 if prefix_ablation:
  # Remove scheme and leading www while retaining path/query so the feature budget
  # is identical. This is an ablation, not a claim of semantic equivalence.
  host=re.sub(r'^www\.','',host)
  original='http://'+host+(p.path or '')+(('?'+p.query) if p.query else '')+(('#'+p.fragment) if p.fragment else '')
  p,host=parse_url(original)
 raw=original.lower();base=raw.split('://',1)[-1];n=max(len(base),1)
 import ipaddress
 try:ipaddress.ip_address(host.strip('[]'));ip=1
 except ValueError:ip=0
 try:port=int(p.port is not None)
 except ValueError:port=0
 return [len(original),len(host),len(p.path),len(p.query),len(p.fragment),host.count('.'),max(0,len(host.split('.'))-2),
         sum(c.isdigit() for c in base),sum(c.isdigit() for c in base)/n,sum(c.isalpha() for c in base)/n,
         base.count('-'),base.count('.'),base.count('@'),base.count('='),base.count('?'),base.count('&'),
         base.count('%'),base.count('/'),base.count('_'),base.count('~'),base.count(':'),base.count('#'),
         base.count(';'),int(p.scheme.lower()=='https'),ip,int('xn--' in host),port,shannon(base),shannon(host),
         len([a for a in re.split(r'[^a-z0-9]+',base) if a]),host.count('-')]

def domain_of(host, extractor):
 if not host:return 'invalid.local'
 result=extractor(host)
 return (result.domain+'.'+result.suffix).lower() if result.suffix and result.domain else host

def extract_frame(frame,ext):
 n=len(frame);x=np.empty((n,len(F_NAMES)),dtype=np.float32);xp=np.empty_like(x);g=np.empty(n,dtype=object)
 for i,u in enumerate(frame.URL):
  x[i]=feature_row(u);xp[i]=feature_row(u,True)
  _,host=parse_url(u);g[i]=domain_of(host,ext)
  if (i+1)%100000==0:print('Parsed',i+1,'of',n,flush=True)
 assert np.isfinite(x).all() and np.isfinite(xp).all()
 return x,xp,g

def hash_fraction(s,seed):
 b=hashlib.sha256(f'{seed}|{s}'.encode('utf-8')).digest()
 return int.from_bytes(b[:8],'big')/(2**64)

def split_global(groups,seed=SEED,cal_fraction=.18):
 is_cal=np.fromiter((hash_fraction(str(x),seed)<cal_fraction for x in groups),dtype=bool,count=len(groups))
 return ~is_cal,is_cal

def cap_samples(d,seed=SEED):
 """Cap each class per domain and corpus; then cap total rows per class/corpus."""
 rng=np.random.default_rng(seed)
 sampled=[]
 for (name,y,dom), group in d.groupby(['corpus','y','domain'],sort=False):
  ix=group.index.to_numpy();sampled.extend(rng.choice(ix,size=min(len(ix),MAX_CLASS_DOMAIN),replace=False).tolist())
 out=d.loc[sampled].copy()
 selected=[]
 for (_, _),group in out.groupby(['corpus','y'],sort=False):
  ids=group.index.to_numpy();selected.extend(rng.choice(ids,size=min(len(ids),MAX_CLASS_SOURCE),replace=False).tolist())
 return out.loc[selected].sort_index().reset_index(drop=True)

def source_min_counts(d,origin_col='corpus'):
 if d.empty:return pd.DataFrame()
 rows=[]
 for name,part in d.groupby(origin_col):
  rows.append({'corpus':name,'n':len(part),'n_benign':int((part.y==0).sum()),'n_phishing':int((part.y==1).sum()),
               'groups_benign':int(part.loc[part.y==0,'domain'].nunique()),'groups_phishing':int(part.loc[part.y==1,'domain'].nunique())})
 return pd.DataFrame(rows)

def roc_direction(x,y):
 if len(np.unique(y))<2:return .5
 # AUC of raw feature. Rank-based test is direction-sensitive.
 if np.max(x)==np.min(x):return .5
 return float(roc_auc_score(y,x))

def stability_scores(d,X,seed=SEED,lambda_=0.1,kind='ccps'):
 """Return per-feature stable discrimination A, conditional source KS B and score.

 The sign is chosen by pooled training data ONLY. No target data used.
 Source class samples are bounded for computational stability.
 """
 nfeat=X.shape[1];rng=np.random.default_rng(seed)
 src=sorted(d.corpus.unique());y=d.y.to_numpy()
 pooled=np.array([roc_direction(X[:,j],y) for j in range(nfeat)])
 signs=np.where(pooled>=.5,1.,-1.)
 aucs=np.zeros((len(src),nfeat))
 for k,name in enumerate(src):
  idx=np.where(d.corpus.to_numpy()==name)[0]
  for j in range(nfeat):aucs[k,j]=roc_direction(X[idx,j],y[idx])
 a_min=np.maximum(0.,np.min((aucs-.5)*signs,axis=0))
 a_mean=np.maximum(0.,np.mean((aucs-.5)*signs,axis=0))
 B=np.zeros(nfeat)
 for yi in (0,1):
  arr={}
  for name in src:
   idx=np.where((d.corpus.to_numpy()==name)&(y==yi))[0]
   if len(idx)>KS_SAMPLE:idx=rng.choice(idx,size=KS_SAMPLE,replace=False)
   arr[name]=idx
  for i,left in enumerate(src):
   for right in src[i+1:]:
    if min(len(arr[left]),len(arr[right]))<20:continue
    for j in range(nfeat):
     kval=float(ks_2samp(X[arr[left],j],X[arr[right],j],method='asymp').statistic)
     B[j]=max(B[j],kval)
 if kind=='ccps':score=a_min-lambda_*B
 elif kind=='stability':score=a_min
 elif kind=='provenance':score=-B
 elif kind=='mean_ccps':score=a_mean-lambda_*B
 else:raise ValueError(kind)
 return {'score':score,'A_min':a_min,'A_mean':a_mean,'B':B,'sign':signs,'aucs':aucs,'sources':src}

def mi_scores(d,X,seed):
 y=d.y.to_numpy();rng=np.random.default_rng(seed)
 ids=np.arange(len(y))
 if len(ids)>MI_SAMPLE:
  ix=[]
  for cls in (0,1):
   a=ids[y==cls];ix.extend(rng.choice(a,size=min(len(a),MI_SAMPLE//2),replace=False).tolist())
  ids=np.asarray(ix,dtype=int)
 return mutual_info_classif(X[ids],y[ids],random_state=seed,n_neighbors=3)

def lr_fit(X,y):
 scaler=StandardScaler();a=scaler.fit_transform(X)
 model=LogisticRegression(max_iter=350,solver='liblinear',class_weight='balanced',random_state=SEED)
 model.fit(a,y)
 return scaler,model

def lr_predict(fit,X):return fit[1].predict_proba(fit[0].transform(X))[:,1]

def threshold_maxcal(model,rep,cal_df,Xcal,at_fpr=CAL_FPR):
 prob=lr_predict(model,Xcal[:,rep]) if isinstance(model,tuple) else model.predict_proba(Xcal[:,rep])[:,1]
 thresholds={};parts={}
 for source in sorted(cal_df.corpus.unique()):
  z=(cal_df.corpus==source).to_numpy() & (cal_df.y.to_numpy()==0)
  values=prob[z]
  if len(values)<50:continue
  t=float(np.nextafter(np.quantile(values,1-at_fpr,method='higher'),np.inf))
  thresholds[source]=t;parts[source]=int(len(values))
 if not thresholds:raise ValueError('No source with sufficient benign calibration records.')
 return min(1.000001,max(thresholds.values())),thresholds,parts

def stats(y,score,th):
 y=np.asarray(y,dtype=np.int8);p=np.asarray(score,dtype=np.float64)
 yp=(p>=th).astype(np.int8)
 tn,fp,fn,tp=confusion_matrix(y,yp,labels=[0,1]).ravel()
 return {'n':len(y),'TP':int(tp),'FN':int(fn),'TN':int(tn),'FP':int(fp),
         'FPR':float(fp/(fp+tn)) if (fp+tn)>0 else None,
         'Recall':float(tp/(tp+fn)) if (tp+fn)>0 else None,
         'MCC':float(matthews_corrcoef(y,yp)) if len(y) else None,
         'Accuracy':float(np.mean(y==yp)) if len(y) else None,
         'PR_AUC':float(average_precision_score(y,p)) if np.unique(y).size==2 else None}

def exact_ci(k,n,alpha=.05):
 if n==0:return [None,None]
 return [float(beta.ppf(alpha/2,k,n-k+1)) if k else 0.,float(beta.ppf(1-alpha/2,k+1,n-k)) if k<n else 1.]

def selector_methods(d,X,k,lam,seed):
 stab=stability_scores(d,X,seed,lam)
 mi=mi_scores(d,X,seed)
 def pick(scores,kk):return np.argsort(-np.asarray(scores),kind='stable')[:kk]
 return {
  'full':np.arange(X.shape[1]),
  'MI':pick(mi,k),
  'stability_only':pick(stab['A_min'],k),
  'provenance_only':pick(-stab['B'],k),
  'mean_CCPS':pick(stab['A_mean']-lam*stab['B'],k),
  'CCPS':pick(stab['score'],k),
  'prefix_ablation':np.arange(X.shape[1])
 },stab,mi

def tune_ccps(train_df,cal_df,X_train,X_cal,seed,logs):
 """Inner leave-one-training-source-out; each source is a validation fold.

 Feature scores are computed once per held source. No target-corpus data is read.
 """
 src=sorted(train_df.corpus.unique());cache={}
 for held in src:
  train_sources=[s for s in src if s!=held]
  a=train_df[train_df.corpus.isin(train_sources)].copy()
  h=train_df[train_df.corpus==held].copy()
  blocked=set(a.domain)|set(cal_df.loc[cal_df.corpus.isin(train_sources),'domain'])
  h=h[~h.domain.isin(blocked)]
  if len(train_sources)<2 or len(h)<2*MIN_EVAL_CLASS or min((h.y==0).sum(),(h.y==1).sum())<MIN_EVAL_CLASS:continue
  if min(h.loc[h.y==0,'domain'].nunique(),h.loc[h.y==1,'domain'].nunique())<MIN_EVAL_GROUPS:continue
  ma=train_df.index.isin(a.index);mh=train_df.index.isin(h.index)
  mc=cal_df.corpus.isin(train_sources).to_numpy()
  c=cal_df[mc]
  if len(c)<100 or np.unique(c.y).size<2:continue
  if min((a.y==0).sum(),(a.y==1).sum())<MIN_TRAIN_CLASS:continue
  xa=X_train[ma];xh=X_train[mh];xc=X_cal[mc]
  try:
   base=stability_scores(a.reset_index(drop=True),xa,seed,0.)
   cache[held]=(xa,a.y.to_numpy(),xh,h.y.to_numpy(),xc,c,base)
  except (ValueError,RuntimeError) as e:logs.append({'scope':'inner-score','held_source':held,'error':str(e)})
 results=[]
 for lam,k in itertools.product(LAMBDA_VALUES,K_VALUES):
  scores=[]
  for held,(xa,ya,xh,yh,xc,c,base) in cache.items():
   rep=np.argsort(-(base['A_min']-lam*base['B']),kind='stable')[:k]
   try:
    model=lr_fit(xa[:,rep],ya)
    threshold,_,_=threshold_maxcal(model,np.arange(len(rep)),c,xc[:,rep])
    met=stats(yh,lr_predict(model,xh[:,rep]),threshold)
    scores.append(float(met['Recall']-met['FPR']))
   except (ValueError,RuntimeError) as e:
    logs.append({'scope':'inner','held_source':held,'lambda':lam,'k':k,'error':str(e)})
  results.append({'lambda':lam,'k':k,'n_valid_source_holdouts':len(scores),
                  'worst_heldout_youden':min(scores) if scores else None,
                  'mean_heldout_youden':float(np.mean(scores)) if scores else None})
 feasible=[a for a in results if a['n_valid_source_holdouts']>=1]
 if feasible:
  best=max(feasible,key=lambda x:(x['n_valid_source_holdouts'],x['worst_heldout_youden'],x['mean_heldout_youden'],-x['k'],-x['lambda']))
 else:
  best={'lambda':0.1,'k':16,'n_valid_source_holdouts':0,'worst_heldout_youden':None,'mean_heldout_youden':None,'fallback_fixed_protocol':True}
 return best,results

def paired_boot(y,groups,p_a,p_b,threshold_a,threshold_b,seed,reps=BOOT_REPS):
 """Paired domain cluster bootstrap of CCPS-minus-baseline FPR and recall."""
 y=np.asarray(y);p_a=np.asarray(p_a);p_b=np.asarray(p_b);g=np.asarray(groups)
 codes,uniques=pd.factorize(g);n=len(uniques)
 if n<MIN_EVAL_GROUPS:return {'valid':False,'reason':'too few independent groups'}
 a=(p_a>=threshold_a);b=(p_b>=threshold_b)
 neg=np.bincount(codes,weights=(y==0).astype(int),minlength=n)
 pos=np.bincount(codes,weights=(y==1).astype(int),minlength=n)
 fa=np.bincount(codes,weights=((y==0)&a).astype(int),minlength=n)
 fb=np.bincount(codes,weights=((y==0)&b).astype(int),minlength=n)
 ta=np.bincount(codes,weights=((y==1)&a).astype(int),minlength=n)
 tb=np.bincount(codes,weights=((y==1)&b).astype(int),minlength=n)
 rng=np.random.default_rng(seed)
 fprs=[];recalls=[]
 for _ in range(reps):
  ids=rng.integers(0,n,size=n)
  ng=neg[ids].sum();pg=pos[ids].sum()
  if ng<10 or pg<10:continue
  fprs.append(float((fa[ids].sum()-fb[ids].sum())/ng))
  recalls.append(float((ta[ids].sum()-tb[ids].sum())/pg))
 if len(fprs)<reps//2:return {'valid':False,'reason':'insufficient two-class bootstrap draws'}
 def interval(z):return [float(v) for v in np.quantile(z,[.025,.5,.975])]
 return {'valid':True,'n_groups':n,'n_reps':len(fprs),'CCPS_minus_baseline_FPR_CI95':interval(fprs),'CCPS_minus_baseline_Recall_CI95':interval(recalls)}

def load_all(pathdict,roots,ext):
 data={};rows=[];paths=[]
 for name in CORPORA:
  p=locate(name,pathdict.get(name),roots)
  raw,path,h=read_raw(p,name)
  base,info=unify(raw,name)
  x,xp,g=extract_frame(base,ext)
  invalid=(g=='invalid.local')
  info['invalid_hostname_removed']=int(invalid.sum())
  if invalid.any():
   base=base.loc[~invalid].reset_index(drop=True);x=x[~invalid];xp=xp[~invalid];g=g[~invalid]
  base['domain']=g;base['corpus']=name
  info.update({'distinct_domains':int(len(set(g))), 'domains_benign':int(len(set(g[base.y==0]))),'domains_phishing':int(len(set(g[base.y==1]))),'file':path,'sha256':h})
  data[name]={'frame':base,'raw':x,'prefix':xp};rows.append(info);paths.append(p)
  print(f'{name}: {len(base):,} records, {len(set(g)):,} distinct registered domains; labels {info["n_legitimate"]} benign/{info["n_phishing"]} phishing',flush=True)
 return data,rows,paths

def build_source(data,target,seed):
 sources=[s for s in CORPORA if s!=target]
 allframes=[]
 for source in sources:
  z=data[source]['frame'].copy()
  z['raw_index']=np.arange(len(z));allframes.append(z)
 frame=pd.concat(allframes,ignore_index=True)
 # conflicting labels between source corpora => all corresponding URLs removed
 conflicted=frame.groupby('URL').y.nunique()
 bad=set(conflicted[conflicted>1].index)
 conflictrows=int(frame.URL.isin(bad).sum())
 frame=frame[~frame.URL.isin(bad)].drop_duplicates('URL',keep='first').copy()
 frame=cap_samples(frame,seed)
 istrain,iscal=split_global(frame.domain,seed)
 tr=frame[istrain].reset_index(drop=True);cal=frame[iscal].reset_index(drop=True)
 # hard check: no domain group may cross train/cal under any corpus
 assert not (set(tr.domain)&set(cal.domain)),'GLOBAL_TRAIN_CAL_GROUP_LEAK'
 def getX(part,view):
  return np.vstack([data[s][view][ix] for s,ix in zip(part.corpus,part.raw_index)]).astype(np.float32) if len(part) else np.empty((0,len(F_NAMES)),dtype=np.float32)
 return tr,cal,getX(tr,'raw'),getX(cal,'raw'),getX(tr,'prefix'),getX(cal,'prefix'),{'source_conflict_rows_removed':conflictrows,'source_post_conflict_dedup_and_cap_removed':int(sum(len(data[s]['frame']) for s in sources)-conflictrows-len(frame)), 'train_count':len(tr),'cal_count':len(cal)}

def assess_quality(frame,original):
 n0=int((original.y==0).sum());n1=int((original.y==1).sum())
 if len(frame)==0:return {'n':0,'quality_primary':False,'retention_benign':0,'retention_phishing':0}
 t0=int((frame.y==0).sum());t1=int((frame.y==1).sum())
 g0=int(frame.loc[frame.y==0,'domain'].nunique());g1=int(frame.loc[frame.y==1,'domain'].nunique())
 r0=t0/max(n0,1);r1=t1/max(n1,1)
 return {'n':len(frame),'n_benign':t0,'n_phishing':t1,'n_groups_benign':g0,'n_groups_phishing':g1,'retention_benign':r0,'retention_phishing':r1,
  'quality_primary':bool(t0>=MIN_EVAL_CLASS and t1>=MIN_EVAL_CLASS and min(g0,g1)>=MIN_EVAL_GROUPS and min(r0,r1)>=MIN_RETENTION)}

def make_figs(out):
 file=out/'Table_Performance.csv'
 if not file.exists():return
 d=pd.read_csv(file)
 focus=d[(d['policy']=='strict_unseen')&(d['model'].isin(['LR_full','LR_MI','LR_CCPS','HGB_full','HGB_CCPS']))].copy()
 focus=focus[focus['quality_primary']==True]
 for metric,title in [('FPR','False-positive rate'),('Recall','Phishing recall'),('MCC','Matthews correlation coefficient')]:
  fig,ax=plt.subplots(figsize=(9.0,4.5))
  if len(focus):
   p=focus.pivot_table(index='target',columns='model',values=metric,aggfunc='mean')
   order=[x for x in CORPORA if x in p.index];p=p.loc[order]
   vals=p.to_numpy();n=len(order);x=np.arange(n);keys=list(p.columns);width=.82/max(len(keys),1)
   for i,key in enumerate(keys):
    ax.bar(x-.41+(i+.5)*width,p[key],width,label=key,facecolor=str(.18+.14*i),edgecolor='black',linewidth=.4,hatch=('/' if i%2 else None))
   ax.set_xticks(x,order);ax.legend(fontsize=7,ncol=min(5,len(keys)))
  else:
   ax.text(.5,.5,'No strict unseen-domain test passed the class-retention quality gate',ha='center',va='center',transform=ax.transAxes)
  ax.set_ylabel(title);ax.set_title(f'External {title} (predeclared eligible targets only)');ax.grid(axis='y',alpha=.2)
  fig.tight_layout();fig.savefig(out/f'Figure_{metric}.png',dpi=300);fig.savefig(out/f'Figure_{metric}.svg');plt.close(fig)

def study(data,out,seed=SEED):
 perf=[];quality=[];tuned=[];fscores=[];calibration=[];boot=[];audit=[];prediction_cache=[]
 for target in CORPORA:
  print('\n'+'='*60+'\nHELD-OUT TARGET:',target,'\n'+'='*60,flush=True)
  tr,cal,xt,xc,xtp,xcp,sourceaudit=build_source(data,target,seed)
  if tr.corpus.nunique()<2 or tr.y.nunique()<2 or cal.y.nunique()<2:
   raise RuntimeError(f'{target}: source train/cal missing two classes or multiple sources')
  audit.append({'target':target,**sourceaudit})
  sourcecounts=source_min_counts(tr);sourcecounts['target']=target;sourcecounts.to_csv(out/f'Training_Source_Composition_{target}.csv',index=False)
  print('Training source counts:\n',sourcecounts.to_string(index=False))
  logs=[]
  tune,grid=tune_ccps(tr,cal,xt,xc,seed,logs)
  tuned.extend([{'target':target,**r} for r in grid]);print('Selected fixed source-only hyperparameters:',tune,flush=True)
  k=int(tune['k']);lam=float(tune['lambda'])
  selections,stab,mi=selector_methods(tr,xt,k,lam,seed)
  for j,nm in enumerate(F_NAMES):
   fscores.append({'target':target,'feature':nm,'A_min':float(stab['A_min'][j]),'A_mean':float(stab['A_mean'][j]),'conditional_provenance_KS':float(stab['B'][j]),'score_CCPS':float(stab['score'][j]),'mutual_information':float(mi[j]),'selected_CCPS':int(j in selections['CCPS']),'selected_MI':int(j in selections['MI'])})
  models={}
  for name,ix in selections.items():
   XX=xtp if name=='prefix_ablation' else xt
   if name=='prefix_ablation':Xc=xcp
   else:Xc=xc
   fit=lr_fit(XX[:,ix],tr.y.to_numpy())
   th,per,counts=threshold_maxcal(fit,np.arange(len(ix)),cal,Xc[:,ix],CAL_FPR)
   models['LR_'+name]=(fit,ix,th,name,'LR')
   calibration.append({'target':target,'model':'LR_'+name,'threshold':th,'source_thresholds':json.dumps(per),'source_benign_counts':json.dumps(counts),'selected_features':json.dumps([F_NAMES[i] for i in ix]),'lambda':lam,'k':k,'inner_valid_folds':tune['n_valid_source_holdouts']})
  for name in ['full','CCPS']:
   ix=selections[name]
   model=HistGradientBoostingClassifier(max_iter=115,max_leaf_nodes=15,learning_rate=.08,l2_regularization=1.,min_samples_leaf=45,random_state=seed)
   model.fit(xt[:,ix],tr.y.to_numpy(),sample_weight=compute_sample_weight('balanced',tr.y.to_numpy()))
   th,per,counts=threshold_maxcal(model,np.arange(len(ix)),cal,xc[:,ix],CAL_FPR)
   models['HGB_'+name]=(model,ix,th,name,'HGB')
   calibration.append({'target':target,'model':'HGB_'+name,'threshold':th,'source_thresholds':json.dumps(per),'source_benign_counts':json.dumps(counts),'selected_features':json.dumps([F_NAMES[i] for i in ix]),'lambda':lam,'k':k,'inner_valid_folds':tune['n_valid_source_holdouts']})
  td=data[target]['frame'];origin=td.copy();seen=set(pd.concat([data[s]['frame'].domain for s in CORPORA if s!=target]).tolist())
  strict=~td.domain.isin(seen).to_numpy()
  policies={'full_external':np.ones(len(td),dtype=bool),'strict_unseen':strict}
  preds={}
  for name,(model,ix,threshold,rep,modeltype) in models.items():
   raw=data[target]['prefix'] if rep=='prefix_ablation' else data[target]['raw']
   prob=lr_predict(model,raw[:,ix]) if modeltype=='LR' else model.predict_proba(raw[:,ix])[:,1]
   preds[name]=(prob,threshold)
   for policy,mask in policies.items():
    sub=td[mask].copy();q=assess_quality(sub,td)
    if not np.any(mask):continue
    met=stats(sub.y.to_numpy(),prob[mask],threshold)
    row={'target':target,'model':name,'policy':policy,'threshold':threshold,'lambda':lam,'feature_budget':len(ix),**q,**met,
         'FPR_CI95':json.dumps(exact_ci(met['FP'],met['FP']+met['TN'])), 'Recall_CI95':json.dumps(exact_ci(met['TP'],met['TP']+met['FN']))}
    perf.append(row)
  for policy,mask in policies.items():
   q=assess_quality(td[mask],td);quality.append({'target':target,'policy':policy,**q})
   if q['quality_primary']:
    for baseline in ['LR_full','LR_MI','LR_stability_only','HGB_full']:
     a,ta=preds['LR_CCPS'];b,tb=preds[baseline]
     try:bs=paired_boot(td.y.to_numpy()[mask],td.domain.to_numpy()[mask],a[mask],b[mask],ta,tb,seed+13)
     except Exception as e:bs={'valid':False,'reason':str(e)}
     boot.append({'target':target,'policy':policy,'contrast':'LR_CCPS minus '+baseline,**bs})
  if logs:
   pd.DataFrame(logs).to_csv(out/f'Tuning_Exceptions_{target}.csv',index=False)
  print('Strict quality:',quality[-1],flush=True)
  # Minimal hash-only prediction manifest, not raw URLs or URLs that may carry secrets.
  prediction_cache.append({'target':target,'n_test':len(td),'n_strict':int(strict.sum()),'selected_CCPS':','.join(F_NAMES[z] for z in selections['CCPS'])})
 pd.DataFrame(perf).to_csv(out/'Table_Performance.csv',index=False)
 pd.DataFrame(quality).to_csv(out/'Table_ClassRetention.csv',index=False)
 pd.DataFrame(tuned).to_csv(out/'Table_InnerSourceLOSO_Tuning.csv',index=False)
 pd.DataFrame(fscores).to_csv(out/'Table_FeatureStability.csv',index=False)
 pd.DataFrame(calibration).to_csv(out/'Table_ThresholdCalibration.csv',index=False)
 pd.DataFrame(audit).to_csv(out/'Table_SourceCleaning.csv',index=False)
 pd.DataFrame(boot).to_json(out/'Paired_Domain_Bootstrap.json',orient='records',indent=2)
 make_figs(out)
 return perf,quality,boot,prediction_cache

def write_decision(out,perf,quality,boot,runinfo):
 p=pd.DataFrame(perf);q=pd.DataFrame(quality)
 strict=p[(p.policy=='strict_unseen')&(p.quality_primary==True)]
 valid_targets=sorted(strict.target.unique())
 eligible=len(valid_targets)
 good=[];bad=[]
 for target in valid_targets:
  rows=strict[strict.target==target].set_index('model')
  if not set(['LR_CCPS','LR_MI','LR_full','LR_stability_only']).issubset(set(rows.index)):continue
  cc=rows.loc['LR_CCPS'];base=rows.loc['LR_MI']
  if cc.FPR<base.FPR and cc.MCC>base.MCC and cc.Recall>=base.Recall-0.03:good.append(target)
  else:bad.append(target)
 summary={'status':'FINISHED_NOT_A_PUBLICATION_DECISION','n_valid_strict_targets':eligible,'valid_strict_targets':valid_targets,
          'targets_better_vs_MI_heuristic':good,'targets_not_better_vs_MI_heuristic':bad,
          'preliminary_direction_only':('promising' if eligible>=2 and len(good)>=2 else 'no_clear_multi_target_advantage'),
          'caveat':'Do not claim statistical superiority, causal validity, novelty or production-level FPR based on this heuristic; inspect CIs, target coverage and all baselines.'}
 (out/'EVIDENCE_GATE.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
 with open(out/'REPORT_FA.md','w',encoding='utf-8') as f:
  f.write('# گزارش آزمایش جامع CCPS-URL\n\n')
  f.write('**نتایج فقط از داده‌های واقعی قابل تفسیرند؛ اجرای مصنوعی برای کنترل کد، شاهد علمی نیست.**\n\n')
  f.write('آموزش و تنظیم آستانه صرفاً با سه منبع آموزشی انجام می‌شود و منبع هدف در هر دور کاملاً کنار گذاشته می‌شود.\n')
  f.write('پروتکل: چهار مجموعه داده، دو مدل سبک، انتخاب ویژگی درون‌منبعی و بین‌منبعی، حذف سخت‌گیرانه هم‌پوشانی، بوت‌استرپ گروهی.\n\n')
  f.write('## کیفیت پوشش کلاس‌ها\n\n```\n'+q.to_string(index=False)+'\n```\n\n')
  f.write('## عملکرد در آزمون‌های قابل تفسیر\n\n')
  if len(strict):f.write('```\n'+strict[['target','model','FPR','Recall','MCC','n','n_benign','n_phishing']].to_string(index=False)+'\n```\n\n')
  else:f.write('هیچ هدفی معیار از پیش تعیین‌شده حفظ هر دو کلاس را در سخت‌گیرانه‌ترین پالایش برآورده نکرد.\n\n')
  f.write('## تصمیم خودکار اولیه (صرفاً غربالگری)\n\n```json\n'+json.dumps(summary,ensure_ascii=False,indent=2)+'\n```\n')
  f.write('\n**نوآوری CCPS اثبات‌شده تلقی نمی‌شود.** مقاله‌های یادگیری ناوردای شرطی و انتخاب ویژگی مقاوم باید در Discussion صریح ذکر شوند.\n')
 return summary

def run(args):
 start=time.time();out=Path(args.out).resolve();out.mkdir(parents=True,exist_ok=True)
 try:import tldextract
 except ImportError as e:raise RuntimeError('tldextract is REQUIRED for public suffix registered-domain splitting. Install tldextract==5.3.0; do not substitute naive split(".").') from e
 extractor=tldextract.TLDExtract(suffix_list_urls=(),cache_dir=str(out/'tld_cache'))
 roots=args.search_roots.split(';') if args.search_roots else ['/kaggle/input','/kaggle/working']
 pathdict={'PhiUSIIL':args.phiusiil,'URL-Phish':args.urlphish,'LegitPhish':args.legitphish,'CompPhish':args.compphish}
 data,manifest,files=load_all(pathdict,roots,extractor)
 pd.DataFrame(manifest).to_csv(out/'Table_DatasetManifest.csv',index=False)
 # Between-corpus URL conflicts are AUDITED, never resolved by looking at heldout labels.
 collisions=[]
 for a,b in itertools.combinations(CORPORA,2):
  da=data[a]['frame'][['URL','y']];db=data[b]['frame'][['URL','y']]
  m=da.merge(db,on='URL',suffixes=('_a','_b'))
  collisions.append({'corpus_a':a,'corpus_b':b,'exact_url_overlap':len(m),'conflicting_labels':int((m.y_a!=m.y_b).sum()),'domain_overlap':len(set(data[a]['frame'].domain)&set(data[b]['frame'].domain))})
 pd.DataFrame(collisions).to_csv(out/'Table_SourceOverlap.csv',index=False)
 perf,quality,boot,card=study(data,out,args.seed)
 runinfo={'seed':args.seed,'time_seconds':round(time.time()-start,1),'data':manifest,'params':{'fixed_fpr_calibration':CAL_FPR,'class_domain_cap':MAX_CLASS_DOMAIN,'class_source_cap':MAX_CLASS_SOURCE,'lambda':LAMBDA_VALUES,'k':K_VALUES,'primary_quality_retention':MIN_RETENTION,'bootstrap_reps':BOOT_REPS},'libraries':{'pandas':pd.__version__,'numpy':np.__version__,'tldextract':getattr(tldextract,'__version__','not_provided')},'target_fold_summary':card}
 (out/'RUN_MANIFEST.json').write_text(json.dumps(runinfo,indent=2,default=str),encoding='utf-8')
 verdict=write_decision(out,perf,quality,boot,runinfo)
 (out/'RUN_STATUS.txt').write_text(f'EXECUTED_FOUR_CORPUS_CCPS\nElapsed_seconds={round(time.time()-start,1)}\nTargets={"|".join(CORPORA)}\nSee EVIDENCE_GATE.json and Table_ClassRetention.csv.\n',encoding='utf-8')
 with zipfile.ZipFile(out.parent/'CCPS_URL_ScientiaIranica_results.zip','w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
  for file in out.rglob('*'):
   if file.is_file() and file.name not in ['PublicSuffix.cache'] and 'tld_cache' not in file.parts:z.write(file,arcname=file.relative_to(out))
 print('COMPLETE:',out.parent/'CCPS_URL_ScientiaIranica_results.zip','\nVerdict:',verdict,flush=True)


In [ ]:
# Single comprehensive execution
import argparse
run(argparse.Namespace(
    phiusiil=DATA_PATHS['PhiUSIIL'],
    urlphish=DATA_PATHS['URL-Phish'],
    legitphish=DATA_PATHS['LegitPhish'],
    compphish=DATA_PATHS['CompPhish'],
    out=OUTPUT_DIR, seed=SEED_FIXED, search_roots='/kaggle/input;/kaggle/working'
))

In [ ]:
# Display observed results: DO NOT infer acceptance or originality from automatic pass/fail flags.
import json
from pathlib import Path
from IPython.display import display
print(json.dumps(json.loads((Path(OUTPUT_DIR)/'EVIDENCE_GATE.json').read_text()),ensure_ascii=False,indent=2))
display(pd.read_csv(Path(OUTPUT_DIR)/'Table_ClassRetention.csv'))
display(pd.read_csv(Path(OUTPUT_DIR)/'Table_Performance.csv').query('policy == "strict_unseen"')[['target','model','n','quality_primary','FPR','Recall','MCC','PR_AUC']])
print('Download: /kaggle/working/CCPS_URL_ScientiaIranica_results.zip')